In [9]:
import importlib
import NeuralNetwork
import funcs
import plots

importlib.reload(NeuralNetwork)
importlib.reload(funcs)
importlib.reload(plots)
from NeuralNetwork import NeuralNetwork

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm
import copy
import os
importlib.reload(funcs)

<module 'funcs' from 'c:\\Users\\fabio\\vu_uni\\machine learning\\actual files\\Project-ML\\funcs.py'>

In [10]:
import importlib
import setup
importlib.reload(setup)
from setup import HIDDEN_LAYERS, BATCH_SIZE, MAX_CLUSTERS, PHASE2_MIN_NEURONS, PHASE2_MIN_CONNECTIONS

device = setup.get_device()
N_TRAIN_EPOCHS = 5 if device.type == "cuda" else 8
train_dataset, val_dataset, test_dataset, fresh_dataset, train_loader, val_loader, test_loader, fresh_loader = setup.get_dataloaders()

# Pruning parameters
MIN_WIDTH = 10
MAX_PRUNE_ROUNDS = 150
MAX_ALLOWED_ACC_DROP = 0.2
N_RETRAIN_EPOCHS = 8
PRUNE_FRAC = 0.1
PRUNE_CON_FRAC = 0.25
REGROW_FRAC = 0

device being used: cuda
train size: 172800, val size: 48000, fresh size: 19200, test size: 40000


In [11]:
# Create model and train
model = NeuralNetwork(hidden_sizes=HIDDEN_LAYERS, device=device)
baseline_acc = model.train_model(train_loader=train_loader, epochs=N_TRAIN_EPOCHS)

Training:   0%|          | 0/5 [00:00<?, ?epoch/s]

Epoch 1/5:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 2/5:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 3/5:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 4/5:   0%|          | 0/20 [00:00<?, ?batches/s]

Epoch 5/5:   0%|          | 0/20 [00:00<?, ?batches/s]

Validation:   0%|          | 0/3 [00:00<?, ?batches/s]

## Prune Neurons and Retrain

## Hyperparameter Search

In [12]:
# import pandas as pd

# original_params = sum(p.numel() for p in model.parameters())
# search_results = []

# for prune_frac in [0.05, 0.10, 0.15, 0.20]:
#     params = (MAX_PRUNE_ROUNDS, prune_frac, prune_frac * 2, prune_frac * 0.5, N_RETRAIN_EPOCHS, MAX_ALLOWED_ACC_DROP)
#     candidate = funcs.pruning(copy.deepcopy(model), train_loader, val_loader, params, baseline_acc, use_max_rounds=True, mode='full')
#     val_acc = candidate.accuracy(val_loader)
#     n_params = sum(p.numel() for p in candidate.parameters())
#     search_results.append({
#         'prune_frac': prune_frac,
#         'val_acc': round(val_acc, 4),
#         'n_params': n_params,
#         'compression': round(original_params / n_params, 2)
#     })
#     print(search_results[-1])

# best_prune_frac = max(search_results, key=lambda r: r['val_acc'])['prune_frac']
# print(f"\nBest prune_frac: {best_prune_frac}")
# pd.DataFrame(search_results)

In [13]:
# prune_parameters = (MAX_PRUNE_ROUNDS, best_prune_frac, best_prune_frac * 2, best_prune_frac * 0.5, N_RETRAIN_EPOCHS, MAX_ALLOWED_ACC_DROP)


use_max_rounds = False if device.type == "cuda" else True

prune_parameters = (MAX_PRUNE_ROUNDS, PRUNE_FRAC, PRUNE_CON_FRAC, REGROW_FRAC, N_RETRAIN_EPOCHS, MAX_ALLOWED_ACC_DROP)

final_model = funcs.pruning(model, train_loader, val_loader, prune_parameters, baseline_acc, use_max_rounds=False, mode='full', fresh_loader=fresh_loader, min_width=MIN_WIDTH, max_clusters=MAX_CLUSTERS, phase2_min_neurons=PHASE2_MIN_NEURONS, phase2_min_connections=PHASE2_MIN_CONNECTIONS)


--- Pruning round 1 ---


  0%|          | 0/1 [00:00<?, ?it/s]

Pruned neurons: layer_0: -6, layer_2: -6, layer_4: -6, layer_6: -3  →  [58 → 58 → 58 → 29]
Retraining after pruning...


Training:   0%|          | 0/8 [00:00<?, ?epoch/s]

Epoch 1/8:   0%|          | 0/22 [00:00<?, ?batches/s]

...

Epoch 10/15:   0%|          | 0/22 [00:00<?, ?batches/s]

Validation:   0%|          | 0/3 [00:00<?, ?batches/s]

Epoch 11/15:   0%|          | 0/22 [00:00<?, ?batches/s]

Validation:   0%|          | 0/3 [00:00<?, ?batches/s]

Early stopping triggered at epoch 11


In [14]:
print(f"Test accuracy after pruning: {final_model.accuracy(val_loader):.2f}")

  0%|          | 0/6 [00:00<?, ?it/s]

Test accuracy after pruning: 0.95


In [15]:
torch.save(final_model, "pruned_model.pth")